## Notebook 05e -- Seed variance of the MLP and XGBoost (M6)
Every stored MLP and XGBoost number comes from one seed (42). This notebook retrains the same fits with seeds 0, 1, ..., 19 (the pilot, Colas et al. 2018) and measures how much the RMSE moves with the seed alone. Plan: `docs/logs/20261012_181201_m5_m6_seed_and_alpha_plan.md`.
Hyperparameters are **not** searched again. Each fit reuses the stored `best_params` of its fold (fixed-hyperparameter design, Bouthillier et al. 2021). Only the seed changes: `torch.manual_seed` for the MLP (weights, dropout, batch order) and `random_state` for XGBoost (row and column subsampling).
Arms: MLP rung 1 (98 folds), XGBoost rung 1 (98 folds), MLP rung 3 (10 fits), MLP refit and MLP control (10 fits each, weighting as in `05c`). One seed of all arms takes about 41 min (plan §6, task 0.1), so the 20-seed pilot takes about 13.6 h.
Outputs (resumable): `fold_metrics_mlp_seeds.json`, `fold_metrics_xgboost_seeds.json`. Seed 42 is not in the seed list. It stays in the stored files and is reported separately. Statistics are a later phase, in `significance_analysis.py`.
Run headless from the `notebooks/` folder: `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 05e_seed_variance.ipynb`

Load the shared data (same `FEATURES`, `FEATURE_COLS`, `SESSION_FOLDS`, `SUBJECT_FOLDS`, `MANIFEST` as 03/04/05).

In [ ]:
from _common import *

Imports and device. Same stack as `05c_rung4_weighted_refit.ipynb`.

In [ ]:
import gc
import os
import time
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

Switches and paths. `RUN_LOOP = False` stops after the reproduction check. `SMOKE = True` writes to separate files and runs seed 0 for participant 010 only (rung 1: the 9 folds of participant 010).
To add seeds after the pilot (plan §3, step 4), extend `SEEDS` and run again. Seeds already on disk are skipped.

In [ ]:
RUN_LOOP = True
SMOKE = False

SEEDS = [0] if SMOKE else list(range(20))
SMOKE_PARTICIPANT = 10
ENROL_SESSION = 1
assert 42 not in SEEDS                     # seed 42 = the stored runs, reported separately

suffix = '_smoke' if SMOKE else ''
OUT_PATHS = {
    'MLP': CURRENT_FOLDER / f'fold_metrics_mlp_seeds{suffix}.json',
    'XGBoost': CURRENT_FOLDER / f'fold_metrics_xgboost_seeds{suffix}.json',
}
print('RUN_LOOP:', RUN_LOOP, '| SMOKE:', SMOKE, '| seeds:', SEEDS, '->', [p.name for p in OUT_PATHS.values()])

Session lists come from the manifest. Participant 001 has no session 4 and participant 010 has no session 6.

In [ ]:
SESSIONS = {int(p.removeprefix('participant')): sorted(v['sessions']) for p, v in MANIFEST['participants'].items()}
assert sum(len(s) for s in SESSIONS.values()) == 98
assert len(SESSION_FOLDS) == 98 and len(SUBJECT_FOLDS) == 10
for p, sess in SESSIONS.items():
    assert ENROL_SESSION in sess, p
print({p: len(s) for p, s in SESSIONS.items()})

Model builders, copied from `05c_rung4_weighted_refit.ipynb`. The only change: the seed is an argument, not the fixed `RANDOM_SEED`. With `sample_weight=None` the MLP `fit` is the same as in `05_mlp.ipynb`.

In [ ]:
def make_xgboost(params, seed):
    return XGBRegressor(
        **params,
        random_state=seed,
        device='cuda',
        tree_method='hist',
        n_jobs=-1,
    )


class MLPTorch(nn.Module):
    def __init__(self, n_features, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPRegressor:
    def __init__(self, hidden_dims, dropout, lr, weight_decay, batch_size, n_epochs, random_state):
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.random_state = random_state

    def fit(self, X, y, sample_weight=None):
        torch.manual_seed(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        self.model_ = MLPTorch(X.shape[1], self.hidden_dims, self.dropout).to(DEVICE)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss()

        X_t = torch.from_numpy(X).to(DEVICE)
        y_t = torch.from_numpy(y).to(DEVICE)
        w_t = None if sample_weight is None else torch.from_numpy(np.asarray(sample_weight, dtype=np.float32)).to(DEVICE)
        n = len(X_t)

        self.model_.train()
        for epoch in range(self.n_epochs):
            perm = torch.randperm(n, device=DEVICE)
            for start in range(0, n, self.batch_size):
                idx = perm[start:start + self.batch_size]
                opt.zero_grad()
                pred = self.model_(X_t[idx])
                if w_t is None:
                    loss = loss_fn(pred, y_t[idx])
                else:
                    w_b = w_t[idx]
                    loss = (w_b * (pred - y_t[idx]) ** 2).sum() / w_b.sum()   # weighted MSE, equals MSELoss when all w = 1
                loss.backward()
                opt.step()
        del X_t, y_t, w_t
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        device = next(self.model_.parameters()).device
        X_t = torch.from_numpy(X).to(device)
        self.model_.eval()
        with torch.no_grad():
            pred = self.model_(X_t)
        return pred.cpu().numpy()


def hidden_dims_from(width, n_layers):
    dims = []
    w = width
    for _ in range(n_layers):
        dims.append(max(w, 8))
        w = max(w // 2, 8)
    return dims


def make_mlp(params, seed):
    hidden_dims = hidden_dims_from(params['width'], params['n_layers'])
    return Pipeline([
        ('scale', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_dims=hidden_dims,
            dropout=params['dropout'],
            lr=params['lr'],
            weight_decay=params['weight_decay'],
            batch_size=params['batch_size'],
            n_epochs=params['n_epochs'],
            random_state=seed,
        )),
    ])


MAKE = {'MLP': make_mlp, 'XGBoost': make_xgboost}


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Load the stored seed-42 results. They give the `best_params` of every fit and the numbers to compare with.
`RUNG1[model][(participant, test_session)]`, `RUNG3_MLP[participant]`, `V2[(participant, arm)]` for the MLP refit and control of `05c`, `V1_RUNG3[participant]` for the per-session rung-3 scores at enrolment session 1 (`05b`).

In [ ]:
def load_json(name):
    with open(CURRENT_FOLDER / name) as f:
        return json.load(f)


RUNG1 = {
    'MLP': {(r['participant_id'], r['test_session']): r for r in load_json('fold_metrics_mlp.json')},
    'XGBoost': {(r['participant_id'], r['test_session']): r for r in load_json('fold_metrics_xgboost.json')},
}
RUNG3_MLP = {r['participant_id']: r for r in load_json('fold_metrics_mlp_rung3.json')}
RUNG3_XGB = {r['participant_id']: r for r in load_json('fold_metrics_xgboost_rung3.json')}
V2 = {(r['participant_id'], r['arm']): r for r in load_json('fold_metrics_rung4_v2.json') if r['model'] == 'MLP'}
V1_RUNG3 = {r['participant_id']: r for r in load_json('fold_metrics_rung4.json')
            if r['model'] == 'MLP' and r['enrol_session'] == ENROL_SESSION}

for m in RUNG1:
    assert set(RUNG1[m]) == {(p, s) for p, sess in SESSIONS.items() for s in sess}, m
assert set(RUNG3_MLP) == set(SESSIONS) and len(V2) == 20 and set(V1_RUNG3) == set(SESSIONS)
print({m: len(v) for m, v in RUNG1.items()}, '| rung 3:', len(RUNG3_MLP), '| v2 MLP records:', len(V2))

Scoring helpers. `score_sessions` and `pooled_rmse` are copied from `05c`. Rung 3 is scored two ways from the same predictions: `metrics` over all sessions (the rung-3 number) and `test` per session (for the refit vs rung-3 comparison over sessions 2 to K).

In [ ]:
def rmse_of(pred, y):
    return float(np.sqrt(np.mean((np.asarray(pred, dtype=float) - np.asarray(y, dtype=float)) ** 2)))


def score_sessions(pred, y, sess, test_sessions):
    entries = []
    for s in test_sessions:
        m = sess == s
        err = pred[m] - y[m]
        entries.append({
            'session': int(s),
            'n': int(m.sum()),
            'rmse': float(np.sqrt(np.mean(err ** 2))),
            'mae': float(np.mean(np.abs(err))),
        })
    return entries


def pooled_rmse(entries):
    return float(np.sqrt(sum(e['n'] * e['rmse'] ** 2 for e in entries) / sum(e['n'] for e in entries)))


def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

One function per arm. Each returns one record. The masks and leak guards are the same as in `05_mlp` (rung 1, rung 3) and `05c` (refit, control).

In [ ]:
def run_rung1(model_name, fold, seed):
    p_num, test_session = fold['participant_id'], fold['test_session']
    params = RUNG1[model_name][(p_num, test_session)]['best_params']
    train_mask = (FEATURES['participant_id'] == p_num) & (FEATURES['session_id'].isin(fold['train_sessions']))
    test_mask = (FEATURES['participant_id'] == p_num) & (FEATURES['session_id'] == test_session)
    assert not (FEATURES.loc[train_mask, 'session_id'] == test_session).any()            # leak guard
    assert set(FEATURES.loc[train_mask, 'session_id'].unique()) == set(fold['train_sessions'])

    model = MAKE[model_name](params, seed)
    model.fit(FEATURES.loc[train_mask, FEATURE_COLS], FEATURES.loc[train_mask, 'pct_mvc_corr'])
    y_pred = np.asarray(model.predict(FEATURES.loc[test_mask, FEATURE_COLS]), dtype=float)
    result = evaluate(FEATURES.loc[test_mask, 'pct_mvc_corr'], y_pred,
                      FEATURES.loc[test_mask, 'phase'], FEATURES.loc[test_mask, 'target_pct'])
    del model
    free_gpu()
    return {'model': model_name, 'arm': 'rung1', 'seed': seed, 'participant_id': p_num,
            'fold': fold['fold'], 'test_session': test_session, 'metrics': result}


def pool_masks(fold):
    test_p = int(fold['test_participant'].removeprefix('participant'))
    train_nums = [int(p.removeprefix('participant')) for p in fold['train_participants']]
    pool_mask = FEATURES['participant_id'].isin(train_nums)
    held_mask = FEATURES['participant_id'] == test_p
    assert test_p not in train_nums                                                     # leak guard
    assert not (FEATURES.loc[pool_mask, 'participant_id'] == test_p).any()
    return test_p, pool_mask, held_mask


def run_rung3(fold, seed):
    test_p, pool_mask, held_mask = pool_masks(fold)
    params = RUNG3_MLP[test_p]['best_params']
    model = make_mlp(params, seed)
    model.fit(FEATURES.loc[pool_mask, FEATURE_COLS].to_numpy(), FEATURES.loc[pool_mask, 'pct_mvc_corr'].to_numpy())
    y_test = FEATURES.loc[held_mask, 'pct_mvc_corr'].to_numpy()
    y_pred = np.asarray(model.predict(FEATURES.loc[held_mask, FEATURE_COLS].to_numpy()), dtype=float)
    result = evaluate(y_test, y_pred, FEATURES.loc[held_mask, 'phase'], FEATURES.loc[held_mask, 'target_pct'])
    sess = FEATURES.loc[held_mask, 'session_id'].to_numpy()
    del model
    free_gpu()
    return {'model': 'MLP', 'arm': 'rung3', 'seed': seed, 'participant_id': test_p,
            'metrics': result, 'test': score_sessions(y_pred, y_test, sess, SESSIONS[test_p])}


def run_weighted(arm, fold, seed):
    test_p, pool_mask, held_mask = pool_masks(fold)
    params = RUNG3_MLP[test_p]['best_params']
    enrol_mask = held_mask & (FEATURES['session_id'] == ENROL_SESSION)
    test_mask = held_mask & (FEATURES['session_id'] != ENROL_SESSION)
    train_mask = pool_mask | enrol_mask
    assert not (train_mask & test_mask).any()                                            # leak guard
    assert set(FEATURES.loc[train_mask & held_mask, 'session_id'].unique()) == {ENROL_SESSION}
    test_sessions = [s for s in SESSIONS[test_p] if s != ENROL_SESSION]

    is_enrol = enrol_mask[train_mask].to_numpy()
    n_enrol = int(is_enrol.sum())
    n_pool = int((~is_enrol).sum())
    w_enrol = n_pool / n_enrol if arm == 'refit' else 1.0
    assert n_pool == V2[(test_p, arm)]['n_pool'] and n_enrol == V2[(test_p, arm)]['n_enrol']
    w = np.where(is_enrol, w_enrol, 1.0)

    model = make_mlp(params, seed)
    model.fit(FEATURES.loc[train_mask, FEATURE_COLS].to_numpy(), FEATURES.loc[train_mask, 'pct_mvc_corr'].to_numpy(),
              scale__sample_weight=w, mlp__sample_weight=w)
    y_test = FEATURES.loc[test_mask, 'pct_mvc_corr'].to_numpy()
    y_pred = np.asarray(model.predict(FEATURES.loc[test_mask, FEATURE_COLS].to_numpy()), dtype=float)
    sess = FEATURES.loc[test_mask, 'session_id'].to_numpy()
    del model
    free_gpu()
    return {'model': 'MLP', 'arm': arm, 'seed': seed, 'participant_id': test_p, 'enrol_session': ENROL_SESSION,
            'w': float(w_enrol), 'n_pool': n_pool, 'n_enrol': n_enrol,
            'test': score_sessions(y_pred, y_test, sess, test_sessions)}

### Reproduction check -- seed 42
Seed 42 must give the stored numbers to 4 decimals, or the run stops. Rung 1: participant 002, session 5, MLP twice (determinism) and XGBoost once. Pooled arms: participant 008 (the smallest pool fit), MLP rung 3, refit and control. About 3 min.

In [ ]:
REPRO_P, REPRO_S, REPRO_POOL_P = 2, 5, 8
repro_fold = next(f for f in SESSION_FOLDS if f['participant_id'] == REPRO_P and f['test_session'] == REPRO_S)
repro_subject_fold = next(f for f in SUBJECT_FOLDS if f['test_participant'] == f'participant{REPRO_POOL_P:03d}')

checks = []
for model_name, n_tries in [('MLP', 2), ('XGBoost', 1)]:
    stored = RUNG1[model_name][(REPRO_P, REPRO_S)]['metrics']['overall']['rmse']
    for _ in range(n_tries):
        checks.append((f'{model_name} rung1 p{REPRO_P} s{REPRO_S}', stored,
                       run_rung1(model_name, repro_fold, 42)['metrics']['overall']['rmse']))

checks.append((f'MLP rung3 p{REPRO_POOL_P}', RUNG3_MLP[REPRO_POOL_P]['metrics']['overall']['rmse'],
               run_rung3(repro_subject_fold, 42)['metrics']['overall']['rmse']))
for arm in ['refit', 'control']:
    checks.append((f'MLP {arm} p{REPRO_POOL_P}', pooled_rmse(V2[(REPRO_POOL_P, arm)]['test']),
                   pooled_rmse(run_weighted(arm, repro_subject_fold, 42)['test'])))

for name, stored, got in checks:
    print(f'{name:22s} stored={stored:.6f}  seed 42 now={got:.6f}')
    if round(got, 4) != round(stored, 4):
        raise RuntimeError(f'{name}: seed 42 gives {got:.6f}, stored {stored:.6f}')
print('reproduction check ok')

### Seed loop
Resumable. Each record key is `(arm, seed, participant, test_session)`. Both files are rewritten after every fit (write then rename). The seed is the outer loop, so a stopped run leaves complete seeds plus one partial seed.
Record layout: rung 1 = the same `metrics` as `fold_metrics_*.json`. Rung 3 = `metrics` over all sessions plus per-session `test`. Refit and control = the `05c` layout (`w`, `n_pool`, `n_enrol`, per-session `test`).

In [ ]:
def _save(model_name, records):
    path = OUT_PATHS[model_name]
    tmp_path = path.with_suffix('.json.tmp')
    with open(tmp_path, 'w') as f:
        json.dump(_json_safe(records), f, indent=2)
    os.replace(tmp_path, path)


def _key(r):
    return (r['arm'], r['seed'], r['participant_id'], r.get('test_session'))


RECORDS = {}
for model_name, path in OUT_PATHS.items():
    if path.exists():
        with open(path) as f:
            RECORDS[model_name] = json.load(f)
    else:
        RECORDS[model_name] = []
DONE = {(m,) + _key(r) for m, recs in RECORDS.items() for r in recs}

n_per_seed = 98 + 98 + 3 * 10
print(f'records on disk: {sum(len(v) for v in RECORDS.values())} of {n_per_seed * len(SEEDS)} (full run)')

In [ ]:
if RUN_LOOP:
    t_loop = time.time()
    for seed in SEEDS:
        t_seed = time.time()
        jobs = []
        for model_name in ['MLP', 'XGBoost']:
            for fold in SESSION_FOLDS:
                if SMOKE and fold['participant_id'] != SMOKE_PARTICIPANT:
                    continue
                jobs.append((model_name, ('rung1', seed, fold['participant_id'], fold['test_session']),
                             lambda m=model_name, f=fold: run_rung1(m, f, seed)))
        for arm in ['rung3', 'refit', 'control']:
            for fold in SUBJECT_FOLDS:
                test_p = int(fold['test_participant'].removeprefix('participant'))
                if SMOKE and test_p != SMOKE_PARTICIPANT:
                    continue
                run = (lambda f=fold: run_rung3(f, seed)) if arm == 'rung3' else (lambda a=arm, f=fold: run_weighted(a, f, seed))
                jobs.append(('MLP', (arm, seed, test_p, None), run))

        n_new = 0
        for model_name, key, run in jobs:
            if (model_name,) + key in DONE:
                continue
            record = run()
            assert _key(record) == key
            RECORDS[model_name].append(record)
            DONE.add((model_name,) + key)
            _save(model_name, RECORDS[model_name])
            n_new += 1
        print(f'seed {seed:2d}: {n_new} new fits, {time.time() - t_seed:.0f} s  (loop total {(time.time() - t_loop) / 3600:.2f} h)')
    print(f'done -- {sum(len(v) for v in RECORDS.values())} records')

### Participant values per seed
Participants are rows. One value per participant and seed, the same value the tests use:
- `rung1` = mean over that participant's folds of the fold RMSE (participant mean, as in the headline).
- `rung3` = RMSE over all sessions of the held-out participant.
- `rung3_2K`, `refit`, `control` = pooled RMSE over sessions 2 to K (the `05c` comparison set).
The tables read the JSON files, so they also work after a restart. Only complete seeds are used.

In [ ]:
def participant_values(records, model_name):
    vals = {}                                       # (arm, seed) -> {participant: value}
    by_seed_p = {}
    for r in records:
        if r['arm'] == 'rung1':
            by_seed_p.setdefault((r['seed'], r['participant_id']), []).append(r['metrics']['overall']['rmse'])
    for (seed, p), v in by_seed_p.items():
        if len(v) == len(SESSIONS[p]):
            vals.setdefault(('rung1', seed), {})[p] = float(np.mean(v))
    for r in records:
        p, seed = r['participant_id'], r['seed']
        if r['arm'] == 'rung3':
            vals.setdefault(('rung3', seed), {})[p] = r['metrics']['overall']['rmse']
            vals.setdefault(('rung3_2K', seed), {})[p] = pooled_rmse([e for e in r['test'] if e['session'] != ENROL_SESSION])
        elif r['arm'] in ('refit', 'control'):
            vals.setdefault((r['arm'], seed), {})[p] = pooled_rmse(r['test'])
    n_p = 1 if SMOKE else len(SESSIONS)
    out = {}
    for (arm, seed), d in vals.items():
        if len(d) == n_p:
            out.setdefault(arm, {})[seed] = d
    return {arm: pd.DataFrame(by_seed).sort_index() for arm, by_seed in out.items()}   # rows = participants, columns = seeds


PV = {m: participant_values(load_json(OUT_PATHS[m].name) if OUT_PATHS[m].exists() else [], m) for m in OUT_PATHS}
for m, arms in PV.items():
    for arm, tab in arms.items():
        print(f'{m} {arm}: {tab.shape[1]} complete seeds')

Seed-42 participant values from the stored files, same definitions. `rung3_2K` at seed 42 uses `rmse_rung3` of `fold_metrics_rung4.json` (enrolment session 1). The two plan values of σb are checked here: 0.641 (rung 1, XGBoost − MLP) and 0.937 (rung 3, XGBoost − MLP).

In [ ]:
P = sorted(SESSIONS)
S42 = {
    ('MLP', 'rung1'): pd.Series({p: np.mean([RUNG1['MLP'][(p, s)]['metrics']['overall']['rmse'] for s in SESSIONS[p]]) for p in P}),
    ('XGBoost', 'rung1'): pd.Series({p: np.mean([RUNG1['XGBoost'][(p, s)]['metrics']['overall']['rmse'] for s in SESSIONS[p]]) for p in P}),
    ('MLP', 'rung3'): pd.Series({p: RUNG3_MLP[p]['metrics']['overall']['rmse'] for p in P}),
    ('XGBoost', 'rung3'): pd.Series({p: RUNG3_XGB[p]['metrics']['overall']['rmse'] for p in P}),
    ('MLP', 'rung3_2K'): pd.Series({p: np.sqrt(sum(t['n'] * t['rmse_rung3'] ** 2 for t in V1_RUNG3[p]['test']) / sum(t['n'] for t in V1_RUNG3[p]['test'])) for p in P}),
    ('MLP', 'refit'): pd.Series({p: pooled_rmse(V2[(p, 'refit')]['test']) for p in P}),
    ('MLP', 'control'): pd.Series({p: pooled_rmse(V2[(p, 'control')]['test']) for p in P}),
}
sb_rung1 = (S42[('XGBoost', 'rung1')] - S42[('MLP', 'rung1')]).std(ddof=1)
sb_rung3 = (S42[('XGBoost', 'rung3')] - S42[('MLP', 'rung3')]).std(ddof=1)
sb_refit_r3 = (S42[('MLP', 'refit')] - S42[('MLP', 'rung3_2K')]).std(ddof=1)
sb_refit_ctrl = (S42[('MLP', 'refit')] - S42[('MLP', 'control')]).std(ddof=1)
assert round(sb_rung1, 3) == 0.641 and round(sb_rung3, 3) == 0.937
print(f'sigma_b at seed 42: rung1 {sb_rung1:.3f} | rung3 {sb_rung3:.3f} | refit-rung3 {sb_refit_r3:.3f} | refit-control {sb_refit_ctrl:.3f}')

Seed mean and seed SD per participant and arm. Participants are rows. `s42` = the stored seed-42 value, for reference only (it is not in the seed mean).

In [ ]:
for m, arms in PV.items():
    for arm, tab in arms.items():
        if tab.shape[1] < 2:
            continue
        summary = pd.DataFrame({'seed mean': tab.mean(axis=1), 'seed SD': tab.std(axis=1, ddof=1),
                                'min': tab.min(axis=1), 'max': tab.max(axis=1), 's42': S42[(m, arm)].reindex(tab.index)})
        print(f'{m} {arm} -- {tab.shape[1]} seeds')
        print(summary.round(3).to_string())
        print('mean over participants:', summary.mean().round(3).to_dict())
        print()

### Task 1.3 -- seed count k (plan §3)
For each test family, `v` = mean over participants of the across-seed variance of the paired difference. The difference is formed per seed index (for example XGBoost seed s − MLP seed s). The seed mean of the difference is the difference of the seed means, so its noise variance is `v / k`.
Rule: `k ≥ v / (0.1025 σb²)`, for 5 % inflation of the paired-difference SD. Rung 3 compares the MLP with XGBoost and Ridge at seed 42 only, so `v` is the MLP variance alone. When the two arms are independent, `v` equals the plan's σM² + σX² for rung 1.
Take k = the maximum over the families. If k ≤ 20, the pilot is the main run. If k > 20, run up to ⌈1.2 k⌉ seeds, cap 60.

In [ ]:
def seed_var_of_diff(tab_a, tab_b=None):
    seeds = sorted(set(tab_a.columns) & (set(tab_b.columns) if tab_b is not None else set(tab_a.columns)))
    d = tab_a[seeds] if tab_b is None else tab_a[seeds] - tab_b[seeds]
    return float(d.var(axis=1, ddof=1).mean()), len(seeds)


families = [
    ('rung1 XGBoost - MLP', ('XGBoost', 'rung1'), ('MLP', 'rung1'), sb_rung1),
    ('rung3 MLP vs fixed', ('MLP', 'rung3'), None, sb_rung3),
    ('refit - rung3 (2..K)', ('MLP', 'refit'), ('MLP', 'rung3_2K'), sb_refit_r3),
    ('refit - control', ('MLP', 'refit'), ('MLP', 'control'), sb_refit_ctrl),
]
k_rows = []
for name, a, b, sb in families:
    if a[1] not in PV[a[0]] or (b is not None and b[1] not in PV[b[0]]):
        continue
    v, n_seeds = seed_var_of_diff(PV[a[0]][a[1]], None if b is None else PV[b[0]][b[1]])
    if n_seeds < 2:
        continue
    k_rows.append({'family': name, 'seeds': n_seeds, 'v': v, 'sigma_b': sb, 'k_min': v / (0.1025 * sb ** 2)})

if k_rows:
    k_table = pd.DataFrame(k_rows)
    print(k_table.round(3).to_string(index=False))
    k = int(np.ceil(k_table['k_min'].max()))
    k_run = k if k <= 20 else min(int(np.ceil(1.2 * k)), 60)
    print(f'k = {k} -> ' + ('pilot of 20 is the main run' if k <= 20 else f'run {k_run} seeds' + (' (cap 60 reached)' if k_run == 60 else '')))